# 03 - New Dataset (BloodMNIST)

Notebook này là phần thực nghiệm bổ sung trên dataset mới BloodMNIST.

Mục tiêu:
- Kiểm tra khả năng chuyển từ STL10 sang BloodMNIST.
- Tuning các siêu tham số để giảm lệch miền dữ liệu.
- So sánh PnP dynamic với baseline cố định và khảo sát `lambda`, `warmup`.


In [ ]:
import sys
import pandas as pd

sys.path.append('..')

from src.utils import get_device, load_feature_file, set_seed, train_clustering

SEED = 42
EPOCHS = 50

set_seed(SEED)
device = get_device()
X, y = load_feature_file('../data/bloodmnist_resnet_features.npy', '../data/bloodmnist_labels.npy', device=device)
device


## Plan

- So sánh baseline PnP dynamic với Fixed-K baseline trên BloodMNIST.
- Sweep `lambda` để xem độ nhạy của ngưỡng split/merge.
- Sweep `warmup` để tìm cấu hình phù hợp hơn với miền dữ liệu mới.


In [ ]:
def run_cfg(name, k0=12, lambda_param=2.0, warmup_epochs=20, enable_split=True, enable_merge=True, epochs=EPOCHS):
    k_star, acc, nmi, ari = train_clustering(
        features=X,
        labels=y,
        k_max=k0,
        device=device,
        epochs=epochs,
        lambda_param=lambda_param,
        warmup_epochs=warmup_epochs,
        enable_split=enable_split,
        enable_merge=enable_merge,
    )
    return {
        'Setting': name,
        'K*': k_star,
        'ACC(%)': round(acc * 100, 2),
        'NMI(%)': round(nmi * 100, 2),
        'ARI(%)': round(ari * 100, 2),
    }


In [ ]:
baseline_df = pd.DataFrame([
    run_cfg('PnP dynamic', k0=12, lambda_param=2.0, warmup_epochs=20, enable_split=True, enable_merge=True),
    run_cfg('Fixed-K baseline', k0=12, lambda_param=2.0, warmup_epochs=20, enable_split=False, enable_merge=False),
])
baseline_df


In [ ]:
lambda_df = pd.DataFrame([
    run_cfg('lambda=1.5', k0=12, lambda_param=1.5, warmup_epochs=20, enable_split=True, enable_merge=True),
    run_cfg('lambda=2.0', k0=12, lambda_param=2.0, warmup_epochs=20, enable_split=True, enable_merge=True),
    run_cfg('lambda=2.5', k0=12, lambda_param=2.5, warmup_epochs=20, enable_split=True, enable_merge=True),
    run_cfg('lambda=3.0', k0=12, lambda_param=3.0, warmup_epochs=20, enable_split=True, enable_merge=True),
])
lambda_df


In [ ]:
warmup_df = pd.DataFrame([
    run_cfg('warmup=0', k0=12, lambda_param=2.0, warmup_epochs=0, enable_split=True, enable_merge=True),
    run_cfg('warmup=10', k0=12, lambda_param=2.0, warmup_epochs=10, enable_split=True, enable_merge=True),
    run_cfg('warmup=20', k0=12, lambda_param=2.0, warmup_epochs=20, enable_split=True, enable_merge=True),
    run_cfg('warmup=30', k0=12, lambda_param=2.0, warmup_epochs=30, enable_split=True, enable_merge=True),
])
warmup_df


## Notes

- Đây là notebook `new dataset` theo đúng cấu trúc nộp bài.
- BloodMNIST được dùng như thực nghiệm bổ sung để kiểm tra lệch miền dữ liệu so với STL10.
- Phần viết tổng hợp có thể lấy từ `docs/ablation_and_bloodmnist_analysis.md`.
